# NR-PANNs 2-branch — Scratch training on Kaggle

**No AudioSet PANNs checkpoints.** Wavegram, Log-Mel, Conv Blocks, FC1, NRE-V3, RGHF and classifier all start from random initialization. Original PANNs repository is used **only for architecture definitions**. Activate GPU T4 in Kaggle Settings and turn Internet on for git clone.


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), 'GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


## 1. Install Python dependencies


In [ ]:
%pip -q install soundfile scipy torchlibrosa librosa h5py


## 2. Clone architecture source (no pretrained weights are downloaded)


In [ ]:
from pathlib import Path
import subprocess
panns_source = Path('/kaggle/working/audioset_tagging_cnn')
if not (panns_source / 'pytorch/models.py').is_file():
    subprocess.run(['git', 'clone', 'https://github.com/qiuqiangkong/audioset_tagging_cnn.git', str(panns_source)], check=True)
assert (panns_source / 'pytorch/models.py').is_file()


## 3. Locate ESC-50 metadata and create workspace symlinks


In [ ]:
from pathlib import Path
input_root = Path('/kaggle/input')
csv_files = list(input_root.rglob('esc50.csv'))
assert csv_files, 'Add ESC-50 under Kaggle Add Input first.'
esc_root = csv_files[0].parent.parent
assert (esc_root / 'audio').is_dir(), f'Missing audio directory in {esc_root}'
workspace = Path('/kaggle/working/PANNS')
(workspace / 'data').mkdir(parents=True, exist_ok=True)
for name, src in [('audioset_tagging_cnn', panns_source), ('data/ESC-50', esc_root)]:
    dest = workspace / name
    if dest.is_symlink() and not dest.exists():
        dest.unlink()
    if not dest.exists():
        dest.symlink_to(src, target_is_directory=True)
assert (workspace / 'data/ESC-50/meta/esc50.csv').is_file()
assert (workspace / 'audioset_tagging_cnn/pytorch/models.py').is_file()
print('Ready. ESC-50:', esc_root)


## 4. Make this scratch folder available in `/kaggle/working`

Upload the ZIP as a Kaggle Input Dataset, or upload the ZIP to the notebook session. This cell searches for it automatically. If you added this project to a cloned GitHub repository, adjust `SCRATCH_DIR` accordingly.


In [ ]:
from pathlib import Path
import zipfile
zip_files = list(Path('/kaggle/input').rglob('nrpanns_2branch_scratch.zip')) + list(Path('/kaggle/working').glob('nrpanns_2branch_scratch.zip'))
if zip_files:
    with zipfile.ZipFile(zip_files[0]) as z:
        z.extractall('/kaggle/working')
SCRATCH_DIR = Path('/kaggle/working/nrpanns_2branch_scratch')
assert (SCRATCH_DIR / 'train.py').is_file(), f'Missing {SCRATCH_DIR}; upload scratch ZIP or clone the project.'
print(SCRATCH_DIR)


## 5. Verify random initialization, then train


In [ ]:
import os, subprocess
cmd = ['python', '-u', 'train.py', '--root', str(workspace), '--fold', '5',
       '--epochs', '150', '--patience', '25', '--batch-size', '4', '--workers', '0',
       '--lr-backbone', '3e-4', '--lr-new', '3e-4', '--no-amp',
       '--output', '/kaggle/working/results/scratch_fold5']
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(SCRATCH_DIR), check=True)


## 6. Examine metrics


In [ ]:
import json
from pathlib import Path
result = Path('/kaggle/working/results/scratch_fold5/fold_5/result.json')
if result.is_file():
    print(json.dumps(json.loads(result.read_text()), indent=2))
else:
    print('Training not yet finished; check history.csv and best.pt in fold_5/')
